In [1]:

!pip -q install gradio pandas numpy scikit-learn plotly openai


In [2]:

import gradio as gr
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ---------------------------------------
# 1. CREATE SYNTHETIC CUSTOMER DATA
# ---------------------------------------
np.random.seed(42)
n = 1200

df = pd.DataFrame({
    "Customer_ID": [f"CUST-{i:04d}" for i in range(1, n + 1)],
    "Recency_Days": np.random.randint(1, 181, n),
    "Purchase_Frequency": np.random.randint(1, 25, n),
    "Total_Spend": np.random.uniform(50, 2500, n).round(2),
    "Average_Order_Value": np.random.uniform(20, 300, n).round(2),
    "Engagement_Score": np.random.randint(1, 101, n),
    "Campaign_Response": np.random.uniform(0, 1, n).round(2),
    "Preferred_Channel": np.random.choice(
        ["Email", "Social Media", "SMS", "Website", "Paid Search"], n
    ),
    "Favourite_Category": np.random.choice(
        ["Fashion", "Electronics", "Beauty", "Home", "Lifestyle"], n
    )
})

# Introduce behavioural patterns for a more useful demo
df.loc[df.index[:240], ["Recency_Days", "Purchase_Frequency",
                        "Total_Spend", "Engagement_Score"]] = np.column_stack([
    np.random.randint(1, 20, 240),
    np.random.randint(12, 25, 240),
    np.random.uniform(1200, 2500, 240),
    np.random.randint(70, 101, 240)
])

df.loc[df.index[240:480], ["Recency_Days", "Engagement_Score"]] = np.column_stack([
    np.random.randint(90, 181, 240),
    np.random.randint(1, 40, 240)
])

# ---------------------------------------
# 2. CUSTOMER SEGMENTATION USING K-MEANS
# ---------------------------------------
features = [
    "Recency_Days", "Purchase_Frequency", "Total_Spend",
    "Average_Order_Value", "Engagement_Score", "Campaign_Response"
]

scaler = StandardScaler()
X = scaler.fit_transform(df[features])

model = KMeans(n_clusters=5, random_state=42, n_init=10)
df["Cluster"] = model.fit_predict(X)

summary = df.groupby("Cluster").agg(
    Recency=("Recency_Days", "mean"),
    Frequency=("Purchase_Frequency", "mean"),
    Spend=("Total_Spend", "mean"),
    Engagement=("Engagement_Score", "mean"),
    Response=("Campaign_Response", "mean")
)

# Assign readable names based on observed behaviour
def assign_segment(row):
    if row["Recency"] > 75 and row["Engagement"] < 45:
        return "At-Risk Customers"
    elif row["Spend"] > df["Total_Spend"].quantile(0.70) and row["Frequency"] > 10:
        return "High-Value Customers"
    elif row["Response"] > 0.60:
        return "Offer-Responsive Customers"
    elif row["Engagement"] > 65:
        return "Highly Engaged Customers"
    return "Occasional Customers"

summary["Segment"] = summary.apply(assign_segment, axis=1)

# Ensure each cluster has a distinct label
used = {}
for cluster in summary.index:
    label = summary.loc[cluster, "Segment"]
    used[label] = used.get(label, 0) + 1
    if used[label] > 1:
        summary.loc[cluster, "Segment"] = f"{label} - Group {used[label]}"

df["Segment"] = df["Cluster"].map(summary["Segment"].to_dict())
segments = sorted(df["Segment"].unique().tolist())

# ---------------------------------------
# 3. SEGMENT SUMMARY
# ---------------------------------------
def executive_overview():
    counts = df["Segment"].value_counts().rename_axis(
        "Segment"
    ).reset_index(name="Customers")

    fig = px.bar(
        counts, x="Segment", y="Customers", color="Segment",
        title="Customer Distribution Across Segments",
        template="plotly_white"
    )
    fig.update_layout(showlegend=False)

    scatter = px.scatter(
        df, x="Purchase_Frequency", y="Total_Spend",
        color="Segment", hover_data=["Customer_ID"],
        title="Customer Spending and Purchase Behaviour",
        template="plotly_white"
    )

    metrics = f"""
    # Executive Overview

    **Customers analysed:** {len(df):,}

    **Behavioural segments:** {df['Segment'].nunique()}

    **Average customer spend:** ${df['Total_Spend'].mean():,.2f}

    **Average engagement:** {df['Engagement_Score'].mean():.1f}/100

    **Average campaign response:** {df['Campaign_Response'].mean()*100:.1f}%

    *All metrics use synthetic demo data.*
    """
    return metrics, fig, scatter

# ---------------------------------------
# 4. SEGMENT INTELLIGENCE
# ---------------------------------------
def segment_analysis(segment):
    data = df[df["Segment"] == segment]

    if data.empty:
        return "No segment selected.", None

    metrics = f"""
    ## {segment}

    - **Customers:** {len(data)}
    - **Average recency:** {data['Recency_Days'].mean():.1f} days
    - **Average purchase frequency:** {data['Purchase_Frequency'].mean():.1f}
    - **Average total spend:** ${data['Total_Spend'].mean():,.2f}
    - **Average order value:** ${data['Average_Order_Value'].mean():,.2f}
    - **Average engagement:** {data['Engagement_Score'].mean():.1f}/100
    - **Campaign response:** {data['Campaign_Response'].mean()*100:.1f}%
    - **Common channel:** {data['Preferred_Channel'].mode()[0]}
    - **Popular category:** {data['Favourite_Category'].mode()[0]}
    """

    fig = px.histogram(
        data, x="Total_Spend", color="Preferred_Channel",
        title="Spending Patterns Within This Segment",
        template="plotly_white"
    )
    return metrics, fig

# ---------------------------------------
# 5. MARKETING STRATEGY AGENT
# ---------------------------------------
def marketing_agent(segment, question, api_key):
    data = df[df["Segment"] == segment]

    if data.empty:
        return "Please select a customer segment."

    recency = data["Recency_Days"].mean()
    spend = data["Total_Spend"].mean()
    frequency = data["Purchase_Frequency"].mean()
    engagement = data["Engagement_Score"].mean()
    response = data["Campaign_Response"].mean() * 100
    channel = data["Preferred_Channel"].mode()[0]
    category = data["Favourite_Category"].mode()[0]

    profile = f"""
    Segment: {segment}
    Customers: {len(data)}
    Average recency: {recency:.1f} days
    Average purchase frequency: {frequency:.1f}
    Average spend: ${spend:.2f}
    Engagement: {engagement:.1f}/100
    Campaign response: {response:.1f}%
    Common channel: {channel}
    Popular category: {category}
    """

    fallback = f"""
    ## Marketing Strategy: {segment}

    **Customer profile**
    {profile}

    **Recommended approach**
    """

    if "At-Risk" in segment or recency > 75:
        fallback += """
        - **Objective:** Reactivate inactive customers.
        - **Campaign:** Personalised win-back emails with relevant product updates.
        - **Offer:** Test a limited-time incentive against a no-discount message.
        - **Message:** "We've found a few new options you may want to explore."
        - **KPIs:** Reactivation rate, repeat purchases, incremental revenue.
        """
    elif "High-Value" in segment:
        fallback += """
        - **Objective:** Retain valuable customers and grow repeat purchases.
        - **Campaign:** VIP previews, early access and complementary product bundles.
        - **Offer:** Exclusive benefits rather than blanket discounts.
        - **Message:** "Enjoy early access to products selected around your interests."
        - **KPIs:** Retention, average order value, repeat purchase rate.
        """
    elif "Offer-Responsive" in segment:
        fallback += """
        - **Objective:** Encourage profitable conversions.
        - **Campaign:** Targeted offers and category-specific bundles.
        - **Offer:** A/B test discount levels and minimum-spend thresholds.
        - **Message:** "Explore a tailored offer on products you may like."
        - **KPIs:** Conversion rate, margin after discounts, revenue per campaign.
        """
    elif "Engaged" in segment:
        fallback += """
        - **Objective:** Convert engagement into purchases.
        - **Campaign:** Product recommendations, new arrivals and helpful content.
        - **Offer:** Test bundles before introducing discounts.
        - **Message:** "Discover more ideas selected around your interests."
        - **KPIs:** Click-to-purchase rate, repeat purchases, conversion rate.
        """
    else:
        fallback += """
        - **Objective:** Encourage the next purchase.
        - **Campaign:** Bestsellers, category guides and follow-up messages.
        - **Offer:** Test a next-purchase incentive against content-only messaging.
        - **Message:** "Explore popular products and recommendations for you."
        - **KPIs:** Second-purchase rate, conversion and campaign engagement.
        """

    fallback += f"""

    **Suggested channel:** {channel}

    **Suggested category:** {category}

    **Measurement:** Run a small test and compare results with a control group.

    *These are hypotheses based on synthetic data, not verified customer preferences.*
    """

    if not api_key or not api_key.strip():
        return "### Rule-based strategy (no LLM key supplied)\n" + fallback

    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key.strip())
        response_obj = client.responses.create(
            model="gpt-4.1-mini",
            input=f"""You are a marketing strategy assistant.
Use this aggregate customer profile to generate a differentiated marketing
strategy. Include campaign, offer, channel, sample message and KPIs.
Do not infer sensitive traits or present assumptions as facts.
Label recommendations as hypotheses and suggest how to test them.

Customer profile:
{profile}

User's question: {question or 'Recommend a strategy for this segment.'}
"""
        )
        return "### LLM-Generated Strategy\n\n" + response_obj.output_text
    except Exception as e:
        return (
            "### LLM request failed — fallback strategy\n"
            f"Check the API key and account access. Details: {str(e)[:250]}\n\n"
            + fallback
        )

# ---------------------------------------
# 6. CUSTOMER EXPLORER
# ---------------------------------------
def explore_customers(segment, min_spend, max_recency):
    data = df.copy()

    if segment != "All Segments":
        data = data[data["Segment"] == segment]

    data = data[
        (data["Total_Spend"] >= min_spend) &
        (data["Recency_Days"] <= max_recency)
    ]

    columns = [
        "Customer_ID", "Segment", "Recency_Days",
        "Purchase_Frequency", "Total_Spend",
        "Engagement_Score", "Campaign_Response",
        "Preferred_Channel", "Favourite_Category"
    ]

    return data[columns].head(250)

# ---------------------------------------
# 7. BUILD THE INTERACTIVE DASHBOARD
# ---------------------------------------
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="rose",
        neutral_hue="slate"
    ),
    title="Customer Segmentation Intelligence"
) as demo:

    gr.Markdown("""
    # Customer Segmentation Intelligence Agent
    ### Discover customer behaviour. Personalise marketing. Improve engagement.

    An interactive machine-learning prototype using synthetic customer data.
    """)

    with gr.Tab("Executive Overview"):
        overview_text = gr.Markdown()
        overview_chart = gr.Plot()
        behaviour_chart = gr.Plot()
        demo.load(
            executive_overview,
            outputs=[overview_text, overview_chart, behaviour_chart]
        )

    with gr.Tab("Segment Intelligence"):
        segment_dropdown = gr.Dropdown(
            choices=segments, value=segments[0],
            label="Select Customer Segment"
        )
        segment_text = gr.Markdown()
        segment_chart = gr.Plot()

        segment_dropdown.change(
            segment_analysis,
            inputs=segment_dropdown,
            outputs=[segment_text, segment_chart]
        )
        demo.load(
            segment_analysis,
            inputs=segment_dropdown,
            outputs=[segment_text, segment_chart]
        )

    with gr.Tab("Marketing Strategy Agent"):
        gr.Markdown("""
        ### Ask for a personalised marketing strategy
        Leave the API key blank to use the built-in rule-based strategy.
        Enter your own OpenAI API key to request an LLM-generated strategy.
        """)
        agent_segment = gr.Dropdown(
            choices=segments, value=segments[0],
            label="Choose Customer Segment"
        )
        api_key = gr.Textbox(
            label="Optional OpenAI API Key",
            type="password",
            placeholder="Leave blank for rule-based recommendations"
        )
        question = gr.Textbox(
            label="Ask the Marketing Agent",
            placeholder="How can we increase repeat purchases without large discounts?",
            lines=2
        )
        generate = gr.Button("Generate Strategy", variant="primary")
        strategy_result = gr.Markdown()

        generate.click(
            marketing_agent,
            inputs=[agent_segment, question, api_key],
            outputs=strategy_result
        )

    with gr.Tab("Customer Explorer"):
        filter_segment = gr.Dropdown(
            choices=["All Segments"] + segments,
            value="All Segments",
            label="Filter by Segment"
        )
        min_spend = gr.Slider(
            0, 2500, value=0, step=25,
            label="Minimum Customer Spend ($)"
        )
        max_recency = gr.Slider(
            1, 180, value=180, step=1,
            label="Maximum Days Since Purchase"
        )
        explore_button = gr.Button("Explore Customers")
        customer_table = gr.Dataframe(
            interactive=False, wrap=True
        )

        explore_button.click(
            explore_customers,
            inputs=[filter_segment, min_spend, max_recency],
            outputs=customer_table
        )
        demo.load(
            explore_customers,
            inputs=[filter_segment, min_spend, max_recency],
            outputs=customer_table
        )

    with gr.Tab("Methodology"):
        gr.Markdown("""
        ### How the system works
        1. Generate synthetic customer transaction and engagement records.
        2. Standardise behavioural variables.
        3. Apply K-Means clustering to discover five customer groups.
        4. Describe clusters using behavioural patterns.
        5. Compare spending, recency, purchase frequency and engagement.
        6. Generate segment-specific marketing strategies.
        7. Present results through an interactive Gradio dashboard.

        **Limitations**
        - The dataset is synthetic and intended for demonstration.
        - Segment labels are simplified descriptions of cluster patterns.
        - LLM recommendations are hypotheses, not proof of customer motivation.
        - Real campaigns should be tested against actual business outcomes.
        """)

demo.launch(share=True, debug=False)


/tmp/ipykernel_7345/1865666999.py:304: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f0c55c1145347c4e8c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
